# Rejection-Sampled SFT for tool-use agent (Stage 1 RL)

Single-iteration offline RL via SFT filtering:
1. Load the current tool-use checkpoint (`gpt_medium_tools_best.pth`)
2. Sample `NUM_PROMPTS` prompts from existing SFT trace questions (balanced across categories)
3. For each prompt, roll out `N_ROLLOUTS` agent trajectories (with sampling)
4. Judge each trajectory with DeepSeek (same rubric as test_agent.ipynb)
5. Keep the top-K trajectories per prompt, filter out low scores + truncated + malformed
6. Merge with the original grounded JSONL → write `tool_traces_grounded_rsft_iterN.jsonl`

Then run the existing `tokenize_tool_sft.ipynb` (point `GROUNDED_FILE` at the new file) and `sft_tool_use.ipynb` to produce `gpt_medium_rsft_iterN_best.pth`.

**Safety rails** (informed by the earlier DPO collapse):
- Training ALWAYS starts from `gpt_medium_tools_best.pth` (fixed base) to prevent compounding drift across iterations.
- Rollouts use the *current best* checkpoint for quality (configurable via `ROLLOUT_CKPT`).
- After each iteration, eval on the held-out 100-prompt set. If any category regresses >0.3, stop and keep previous.

**Prereqs in Colab:**
1. Runtime → T4 GPU
2. Secrets: `DEEPSEEK_API_KEY`, optionally `BRAVE_API_KEY`
3. Drive has `gpt_medium_tools_best.pth`, `tokenizer.json`, and `tool_traces_grounded.jsonl`

**Cost**: ~$0.50–2 per iteration in DeepSeek calls. Runtime: ~60–90 min per iteration on T4.

## 1. Install dependencies

In [ ]:
!pip install -q openai tokenizers tqdm pandas

## 2. Config

Edit `ITER_NUM` and `ROLLOUT_CKPT` between runs. Training ALWAYS starts from `BASE_CKPT` in the SFT notebook (set separately).

In [ ]:
import os, sys

# ---- Iteration config ----
ITER_NUM = 1                        # bump this each iteration
NUM_PROMPTS = 250                   # prompts sampled from existing SFT questions
N_ROLLOUTS = 4                      # trajectories per prompt (higher = better ranking but more $)
KEEP_TOP_K = 2                      # keep top-K per prompt (K=2 of 4 = 50% filter)
MIN_SCORE_THRESHOLD = 3.5           # absolute floor — drop anything below this regardless of rank
DROP_TRUNCATED = True               # agent hit max_steps without Final → exclude

# ---- Sampling temperature (higher = more diverse rollouts) ----
ROLLOUT_TEMPERATURE = 0.9

# ---- Checkpoints ----
# For iter 1: ROLLOUT_CKPT = 'gpt_medium_tools_best.pth'
# For iter 2+: swap to the latest RSFT-trained checkpoint for better rollout quality
ROLLOUT_CKPT = 'gpt_medium_tools_best.pth'

print(f'ITER_NUM={ITER_NUM}  NUM_PROMPTS={NUM_PROMPTS}  N_ROLLOUTS={N_ROLLOUTS}  KEEP_TOP_K={KEEP_TOP_K}')
print(f'ROLLOUT_CKPT={ROLLOUT_CKPT}')

## 3. Clone/pull repo + mount Drive + stage weights

In [ ]:
import subprocess, shutil

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_URL = 'https://github.com/ajencinas/sparkyllm.git'
    REPO_DIR = '/content/sparkyllm'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
    from google.colab import drive, userdata
    drive.mount('/content/drive')
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(REPO_DIR)

# Stage weights
WEIGHTS_DIR = os.path.join(REPO_DIR, 'local_test', 'weights')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if IS_COLAB:
    SPARKYLLM_DRIVE = '/content/drive/MyDrive/sparkyllm'
    CHECKPOINT_DRIVE = os.path.join(SPARKYLLM_DRIVE, 'checkpoints', ROLLOUT_CKPT)
    TOKENIZER_DRIVE  = os.path.join(SPARKYLLM_DRIVE, 'datasets_pretrain', 'tokenizer_out', 'tokenizer.json')
    GROUNDED_DRIVE   = os.path.join(SPARKYLLM_DRIVE, 'datasets_sft', 'tool_traces', 'tool_traces_grounded.jsonl')
    for src, dst in [(CHECKPOINT_DRIVE, ROLLOUT_CKPT),
                     (TOKENIZER_DRIVE, 'tokenizer.json')]:
        dst_path = os.path.join(WEIGHTS_DIR, dst)
        if not os.path.exists(src):
            raise FileNotFoundError(f'Missing on Drive: {src}')
        if not os.path.exists(dst_path):
            print(f'Copying {dst}...')
            shutil.copy2(src, dst_path)
        else:
            print(f'  already staged: {dst}')
    # Output directory for this iteration (on Drive, resumable)
    OUT_DIR = os.path.join(SPARKYLLM_DRIVE, 'datasets_sft', 'rsft', f'iter_{ITER_NUM}')
    os.makedirs(OUT_DIR, exist_ok=True)
    GROUNDED_PATH = GROUNDED_DRIVE
else:
    OUT_DIR = os.path.join(REPO_DIR, 'datasets_sft', 'rsft', f'iter_{ITER_NUM}')
    os.makedirs(OUT_DIR, exist_ok=True)
    GROUNDED_PATH = os.path.join(REPO_DIR, 'datasets_sft', 'tool_traces', 'tool_traces_grounded.jsonl')

TRAJECTORIES_PATH = os.path.join(OUT_DIR, 'trajectories.jsonl')
JUDGED_PATH       = os.path.join(OUT_DIR, 'judged.jsonl')
OUTPUT_GROUNDED   = os.path.join(OUT_DIR, f'tool_traces_grounded_rsft_iter{ITER_NUM}.jsonl')

print(f'OUT_DIR:    {OUT_DIR}')
print(f'GROUNDED:   {GROUNDED_PATH}')
print(f'Rollout weights: {os.path.join(WEIGHTS_DIR, ROLLOUT_CKPT)}')

## 4. Load model, tokenizer, agent runner, DeepSeek judge

In [ ]:
sys.path.insert(0, os.path.join(REPO_DIR, 'local_test'))
sys.path.insert(0, os.path.join(REPO_DIR, 'local_agent'))

# Wire Brave key from Colab Secrets (web_search will use it)
if IS_COLAB:
    try:
        brave = userdata.get('BRAVE_API_KEY')
        if brave:
            os.environ['BRAVE_API_KEY'] = brave
            print('Brave API key loaded')
    except Exception:
        pass

from sparky_model import detect_device, load_model, load_tokenizer, vocab_size_for  # type: ignore
from agent import AgentRunner, AgentResult  # type: ignore
from tools import TOOLS  # type: ignore

TOKENIZER_PATH  = os.path.join(WEIGHTS_DIR, 'tokenizer.json')
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, ROLLOUT_CKPT)
assert os.path.exists(TOKENIZER_PATH), f'missing {TOKENIZER_PATH}'
assert os.path.exists(CHECKPOINT_PATH), f'missing {CHECKPOINT_PATH}'

device = detect_device()
tokenizer = load_tokenizer(TOKENIZER_PATH)
vocab = vocab_size_for(tokenizer)
model = load_model(CHECKPOINT_PATH, vocab, device)
runner = AgentRunner(model, tokenizer, device, max_steps=3, temperature=ROLLOUT_TEMPERATURE)
print(f'Loaded rollout model on {device}. Tools: {list(TOOLS)}')

# ---- DeepSeek judge ----
from openai import OpenAI
if IS_COLAB:
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
else:
    DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY')
assert DEEPSEEK_API_KEY, 'DEEPSEEK_API_KEY not set'
judge_client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com')
print('DeepSeek judge ready')

## 5. Sample prompts from existing SFT traces

Bucketed across categories roughly matching the eval distribution.

In [ ]:
import json, random
from collections import defaultdict, Counter

random.seed(42 + ITER_NUM)  # different sample per iteration

# Weights roughly mirror the eval category distribution, weighted toward
# currently-weak categories (web_search, multi_step).
CATEGORY_WEIGHTS = {
    'single_calculator':  0.20,
    'single_web_search':  0.30,  # weakest category — focus here
    'single_time':        0.05,
    'single_none':        0.10,
    'error_recovery':     0.05,
    'multi_step':         0.30,  # also weak, big lever
}

by_category = defaultdict(list)
with open(GROUNDED_PATH) as f:
    for line in f:
        t = json.loads(line)
        cat = t.get('category', 'unknown')
        # Collapse multi_step subcats into 'multi_step'
        if cat.startswith('multi_step') or cat in ('calc_then_compare', 'calc_then_calc', 'search_then_reason',
                                                    'time_then_calc', 'search_then_search', 'calc_then_time',
                                                    'calc_then_search', 'search_then_calc', 'time_then_search'):
            cat = 'multi_step'
        by_category[cat].append(t['question'])

print(f'Source pool sizes:')
for cat, items in by_category.items():
    print(f'  {cat:22s} {len(items)}')

# Sample NUM_PROMPTS total, weighted by CATEGORY_WEIGHTS
prompts = []
for cat, weight in CATEGORY_WEIGHTS.items():
    target_n = int(NUM_PROMPTS * weight)
    pool = by_category.get(cat, [])
    if not pool:
        continue
    k = min(target_n, len(pool))
    sampled = random.sample(pool, k)
    for q in sampled:
        prompts.append({'category': cat, 'prompt': q})

random.shuffle(prompts)
print(f'\nSampled {len(prompts)} prompts:')
print(dict(Counter(p['category'] for p in prompts)))

## 6. Roll out trajectories

Resumable — if the notebook dies mid-rollout, re-running this cell picks up from where `trajectories.jsonl` left off.

In [ ]:
import time
from dataclasses import asdict
from tqdm.auto import tqdm

# Resume: load existing trajectories if any
existing = []
if os.path.exists(TRAJECTORIES_PATH):
    with open(TRAJECTORIES_PATH) as f:
        for line in f:
            existing.append(json.loads(line))
    print(f'Resuming from {len(existing)} existing trajectories')

done_keys = {(r['prompt_id'], r['rollout_id']) for r in existing}

with open(TRAJECTORIES_PATH, 'a') as fout:
    total_to_do = len(prompts) * N_ROLLOUTS
    pbar = tqdm(total=total_to_do, initial=len(done_keys), desc='rollouts')
    for pid, p in enumerate(prompts):
        for rid in range(N_ROLLOUTS):
            if (pid, rid) in done_keys:
                continue
            t0 = time.time()
            try:
                result = runner.run_turn(p['prompt'])
                rec = {
                    'prompt_id': pid,
                    'rollout_id': rid,
                    'category': p['category'],
                    'prompt': p['prompt'],
                    'final_answer': result.final_answer,
                    'steps': [asdict(s) for s in result.steps],
                    'raw_trace': result.raw_trace,
                    'truncated': result.truncated,
                    'latency_s': round(time.time() - t0, 2),
                    'error': None,
                }
            except Exception as e:
                rec = {
                    'prompt_id': pid, 'rollout_id': rid,
                    'category': p['category'], 'prompt': p['prompt'],
                    'final_answer': '', 'steps': [], 'raw_trace': '',
                    'truncated': False, 'latency_s': round(time.time() - t0, 2),
                    'error': f'{type(e).__name__}: {e}',
                }
            fout.write(json.dumps(rec, ensure_ascii=False) + '\n')
            fout.flush()
            pbar.update(1)
    pbar.close()

# Final load
trajectories = []
with open(TRAJECTORIES_PATH) as f:
    for line in f:
        trajectories.append(json.loads(line))
print(f'\nTotal trajectories: {len(trajectories)}')
print(f'Agent errors: {sum(1 for t in trajectories if t.get("error"))}')
print(f'Truncated:    {sum(1 for t in trajectories if t.get("truncated"))}')

## 7. Judge trajectories with DeepSeek

Same 4-criterion rubric as `test_agent.ipynb`. Parallel via ThreadPoolExecutor. Resumable.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

JUDGE_SYSTEM = """You are evaluating a small (650M parameter) language model running a ReAct-style tool-use agent.

You will receive a user PROMPT, the agent's STEPS (Thought/Action/Input/Result sequence), and the agent's FINAL_ANSWER.

Available tools the agent can call:
- calculator  — arithmetic expressions like 2+3*4
- time        — returns current date/time, no input
- web_search  — factual lookup (Brave/Wikipedia)
- none        — sentinel meaning "answer directly"

Score the agent on each criterion using a 1-5 scale (5 = excellent, 1 = failure):
- tool_choice: did the agent pick the right tool(s), or correctly skip tools?
- tool_input: were the tool inputs well-formed and appropriate? (5 if no tool was called.)
- final_answer_correctness: does FINAL_ANSWER correctly and fully answer the prompt?
- trace_coherence: are the Thoughts sensible and does the trace flow logically? Penalize loops, gibberish, or hallucinated tool outputs.

Reply with JSON only, no markdown fences, in this exact shape:
{\"tool_choice\": N, \"tool_input\": N, \"final_answer_correctness\": N, \"trace_coherence\": N, \"reason\": \"one short sentence\"}"""

def _format_steps(steps):
    if not steps: return '(no tool calls — direct answer)'
    parts = []
    for i, s in enumerate(steps, 1):
        res = (s.get('result') or '')[:300]
        if len(s.get('result') or '') > 300:
            res += '...'
        line = (f"Step {i}:\n  Thought: {s.get('thought','')}\n  Action: {s.get('action','')}\n"
                f"  Input: {s.get('input','')}\n  Result: {res}")
        if s.get('error'):
            line += f"\n  ERROR: {s['error']}"
        parts.append(line)
    return '\n'.join(parts)

def judge_one(rec, max_retries=3):
    user_msg = (f"PROMPT:\n{rec['prompt']}\n\nSTEPS:\n{_format_steps(rec['steps'])}\n\n"
                f"FINAL_ANSWER:\n{rec['final_answer']}\n\n(truncated={rec.get('truncated')})")
    for attempt in range(max_retries):
        try:
            comp = judge_client.chat.completions.create(
                model='deepseek-chat',
                messages=[{'role':'system','content':JUDGE_SYSTEM},{'role':'user','content':user_msg}],
                temperature=0.0, max_tokens=200,
            )
            raw = comp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = raw.strip('`').strip()
                if raw.lower().startswith('json'): raw = raw[4:].strip()
            scores = json.loads(raw)
            for k in ('tool_choice','tool_input','final_answer_correctness','trace_coherence'):
                assert k in scores and 1 <= int(scores[k]) <= 5
                scores[k] = int(scores[k])
            scores.setdefault('reason','')
            return scores
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1.5)
    return {'tool_choice':0,'tool_input':0,'final_answer_correctness':0,'trace_coherence':0,'reason':'judge_error'}

# Resume: load existing judgments
judged_keys = set()
judged_records = {}
if os.path.exists(JUDGED_PATH):
    with open(JUDGED_PATH) as f:
        for line in f:
            r = json.loads(line)
            key = (r['prompt_id'], r['rollout_id'])
            judged_keys.add(key)
            judged_records[key] = r
    print(f'Resuming: {len(judged_keys)} already judged')

to_judge = [t for t in trajectories if (t['prompt_id'], t['rollout_id']) not in judged_keys]
print(f'Judging {len(to_judge)} trajectories...')

NUM_JUDGE_WORKERS = 10

with open(JUDGED_PATH, 'a') as fout:
    pbar = tqdm(total=len(to_judge), desc='judge')
    with ThreadPoolExecutor(max_workers=NUM_JUDGE_WORKERS) as ex:
        futures = {ex.submit(judge_one, t): t for t in to_judge}
        for fut in as_completed(futures):
            t = futures[fut]
            try:
                scores = fut.result()
            except Exception as e:
                scores = {'tool_choice':0,'tool_input':0,'final_answer_correctness':0,'trace_coherence':0,'reason':f'exc: {e}'}
            merged = {**t, **scores}
            fout.write(json.dumps(merged, ensure_ascii=False) + '\n')
            fout.flush()
            pbar.update(1)
    pbar.close()

# Reload all judged
judged = []
with open(JUDGED_PATH) as f:
    for line in f:
        judged.append(json.loads(line))

SCORE_COLS = ['tool_choice','tool_input','final_answer_correctness','trace_coherence']
for r in judged:
    r['avg_score'] = sum(r.get(c, 0) for c in SCORE_COLS) / 4

print(f'\nJudged total: {len(judged)}')
print(f'Mean avg_score: {sum(r["avg_score"] for r in judged)/max(1,len(judged)):.2f}')
print(f'\nScore distribution (by bucket):')
for lo, hi in [(0, 1.5), (1.5, 2.5), (2.5, 3.5), (3.5, 4.5), (4.5, 5.01)]:
    n = sum(1 for r in judged if lo <= r['avg_score'] < hi)
    print(f'  [{lo:.1f}, {hi:.1f}): {n:>4d}  ({100*n/max(1,len(judged)):.1f}%)')

## 8. Filter + merge with original SFT data

For each prompt: take the top-K trajectories, drop anything below `MIN_SCORE_THRESHOLD`, drop truncated/errored. Reformat as SFT trace. Merge with original `tool_traces_grounded.jsonl`.

In [ ]:
# Group by prompt_id, rank, and keep top-K
by_prompt = defaultdict(list)
for r in judged:
    by_prompt[r['prompt_id']].append(r)

kept_trajectories = []
rejected_reasons = Counter()

for pid, group in by_prompt.items():
    # Filter hard-fail first
    eligible = []
    for r in group:
        if r.get('error'):
            rejected_reasons['agent_error'] += 1
            continue
        if DROP_TRUNCATED and r.get('truncated'):
            rejected_reasons['truncated'] += 1
            continue
        if not r.get('final_answer', '').strip():
            rejected_reasons['empty_final'] += 1
            continue
        if r['avg_score'] < MIN_SCORE_THRESHOLD:
            rejected_reasons['below_threshold'] += 1
            continue
        eligible.append(r)
    # Sort by avg_score desc, keep top-K
    eligible.sort(key=lambda r: r['avg_score'], reverse=True)
    kept_trajectories.extend(eligible[:KEEP_TOP_K])
    for r in eligible[KEEP_TOP_K:]:
        rejected_reasons['not_in_top_k'] += 1

print(f'Kept {len(kept_trajectories)} trajectories (from {len(judged)} judged)')
print(f'Retention rate: {100*len(kept_trajectories)/max(1,len(judged)):.1f}%')
print(f'\nRejection reasons:')
for reason, n in rejected_reasons.most_common():
    print(f'  {reason:20s} {n}')

print(f'\nMean avg_score of kept: {sum(r["avg_score"] for r in kept_trajectories)/max(1,len(kept_trajectories)):.2f}')
print(f'\nKept by category:')
print(dict(Counter(r['category'] for r in kept_trajectories)))

In [ ]:
# Convert kept rollouts into SFT-trace format (match tokenize_tool_sft.ipynb schema)
def to_sft_format(r):
    return {
        'question': r['prompt'],
        'category': f"rsft_{r['category']}",  # tag so we can trace origin
        'steps': [{
            'thought': s.get('thought', ''),
            'action':  s.get('action', ''),
            'input':   s.get('input', ''),
            'result':  s.get('result', ''),
        } for s in r['steps']],
        'final': r['final_answer'],
        'quality_score': r['avg_score'],
        'source': f'rsft_iter_{ITER_NUM}',
    }

rsft_traces = [to_sft_format(r) for r in kept_trajectories]

# Guard: drop any trace with empty steps (should be rare — direct-answer traces need steps=[{action:none}])
# The tokenizer requires at least one step. For 'none' direct-answer cases, inject a dummy none step.
clean_rsft = []
for t in rsft_traces:
    if not t['steps']:
        t['steps'] = [{'thought': 'I can answer directly.', 'action': 'none', 'input': '', 'result': ''}]
    clean_rsft.append(t)

# Merge with original grounded SFT traces
original_traces = []
with open(GROUNDED_PATH) as f:
    for line in f:
        original_traces.append(json.loads(line))

merged = original_traces + clean_rsft

# Dedup by question
seen = set()
deduped = []
dup_skipped = 0
for t in merged:
    q = t['question'].strip().lower()
    if q in seen:
        dup_skipped += 1
        continue
    seen.add(q)
    deduped.append(t)

with open(OUTPUT_GROUNDED, 'w') as f:
    for t in deduped:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

print(f'Wrote {len(deduped)} traces to:')
print(f'  {OUTPUT_GROUNDED}')
print(f'  (original: {len(original_traces)}, added: {len(clean_rsft)}, deduped: {dup_skipped})')

In [ ]:
# Promote RSFT output to the canonical SFT location so tokenize_tool_sft.ipynb
# picks it up unchanged. Keeps a timestamped backup for rollback.
from datetime import datetime, timezone

if IS_COLAB:
    CANONICAL_GROUNDED = os.path.join(SPARKYLLM_DRIVE, 'datasets_sft', 'tool_traces', 'tool_traces_grounded.jsonl')
else:
    CANONICAL_GROUNDED = os.path.join(REPO_DIR, 'datasets_sft', 'tool_traces', 'tool_traces_grounded.jsonl')

# Backup current canonical (timestamped, one per run)
if os.path.exists(CANONICAL_GROUNDED):
    ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%SZ')
    backup = CANONICAL_GROUNDED.replace('.jsonl', f'_pre_rsft_iter{ITER_NUM}_{ts}.jsonl')
    shutil.copy2(CANONICAL_GROUNDED, backup)
    print(f'Backed up existing canonical → {backup}')
else:
    print(f'No existing canonical file at {CANONICAL_GROUNDED} — nothing to back up')

# Promote RSFT merged file
shutil.copy2(OUTPUT_GROUNDED, CANONICAL_GROUNDED)
print(f'Promoted:  {OUTPUT_GROUNDED}')
print(f'      → {CANONICAL_GROUNDED}')
print(f'\nReady to run:\n  1. sft_tools/tokenize_tool_sft.ipynb\n  2. sft_tools/sft_tool_use.ipynb (START_FROM="dpo")\n  3. test/test_agent.ipynb on the new checkpoint')

## 9. Next steps

The cell above already promoted the RSFT-merged JSONL to the canonical path and backed up the previous one. Now:

1. **Run `sft_tools/tokenize_tool_sft.ipynb`** — produces a fresh `tool_traces_tokenized.jsonl`.

2. **Run `sft_tools/sft_tool_use.ipynb`** — TRAINING STARTS FROM `gpt_medium_tools_best.pth` (the original base; do NOT start from a previous RSFT iter). Rename output to `gpt_medium_rsft_iter{N}_best.pth` to keep iterations distinct.

3. **Re-run `test/test_agent.ipynb`** on the new checkpoint. Compare to the previous eval run.

4. **If eval improved >0.1 avg AND no category regressed >0.3:** bump `ITER_NUM`, set `ROLLOUT_CKPT` to the new RSFT checkpoint, re-run this notebook for iter 2.

5. **If not:** revert to previous checkpoint. Rollback is easy — the backup file `tool_traces_grounded_pre_rsft_iter{N}_{ts}.jsonl` on Drive can be copied back over the canonical name.